In [32]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
from collections import deque
import random 
import pickle 
import os
import zipfile

In [103]:
class memory(nn.Module):
    def __init__(self, vocab_size, hidden_size, embedding_dim=10, layer=0):
        super().__init__()
        self.layer = layer
        self.vocab_size = vocab_size

        if self.layer == 0:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.encoder = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        else:
            self.encoder = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.decoder = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ### 
        for name, param in self.encoder.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
        
        for name, param in self.decoder.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x):
        if self.layer == 0:
            embedded = self.embedding(x)
            _, h = self.encoder(embedded)
        else:
            _, h = self.encoder(x)
        #print('here')
        h_en = h.clone()
        dec_input = torch.zeros((1, 1, self.vocab_size))
        outputs = []
        for _ in range(x.size(1)):
            dec_out, h = self.decoder(dec_input, h)
            logits = self.out(dec_out)
            outputs.append(logits)
            dec_input = logits.detach()
        return torch.cat(outputs, dim=1), h_en.detach()

In [153]:
class prediction(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, context_size=0):
        super().__init__()
        self.context_size = context_size
        self.linear1 = nn.Linear(input_size+context_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)

    def forward(self, h, context=None):
        if self.context_size == 0:
            out_ = F.relu(self.linear1(h))
        else:
            out_ = F.relu(self.linear1(torch.concatenate((h, context), dim=2)))
        
        out = self.linear2(out_)
        return out

In [135]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [136]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
      
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [137]:
def train_layer(model, optimizer, criterion, X, h=None, layer=0, short_term_memory=3):
    #print('Training layer ', layer)
    optimizer.zero_grad()
    #print(X, X.size())
    logits, h = model(X)
    
    loss = sum(criterion(logits[:, t], X[:, t]) for t in range(short_term_memory)) / short_term_memory   
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    return h.detach()
        

In [ ]:
total_samples = 50000000
n_community = 3
n_members = 3

total_layers = 3
context_size = 30
hidden_size_memory = [30, 90, 300]
hidden_size_prediction = 50
vocab_size = n_community*n_members + 1
lr_memory = [1e-3,1e-3,1e-3,1e-3]
short_term_memory = 3

memory_block = {}
input_buffer = {}
h = {}
memory_criteria = []
memory_optimizers = []

for layer in range(total_layers):
    if layer == 0:
        memory_block[layer] = memory(vocab_size, hidden_size_memory[layer], embedding_dim=10, layer=layer)
        memory_criteria.append(
                nn.CrossEntropyLoss()
            )
    else:
        memory_block[layer] = memory(hidden_size_memory[layer-1], hidden_size_memory[layer], layer=layer)
        memory_criteria.append(
                nn.MSELoss()
            )

        input_buffer[layer] = deque(
                [torch.zeros((hidden_size_memory[layer-1])) for _ in range(short_term_memory)],
                maxlen=short_term_memory
            )
    

    memory_optimizers.append(
        torch.optim.Adam(memory_block[layer].parameters(), lr=lr_memory[layer], weight_decay=1e-8)
    )


### handle data ###
#data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)
data = get_random_sequence(total_samples, token_number=vocab_size)
data_set = Dataset_converter(data, 1, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
for X, _ in tqdm(train_loader):
    for layer in range(total_layers):
        downsample = short_term_memory**layer

        if layer == 0:
            h[layer] = train_layer(memory_block[layer], memory_optimizers[layer], memory_criteria[layer], X)

        elif total%(downsample)==0:
            input_buffer[layer].append(
                h[layer-1][0][0].clone()
            )

            h[layer] = train_layer(
                memory_block[layer], memory_optimizers[layer], memory_criteria[layer], 
                torch.stack(list(input_buffer[layer])).unsqueeze(0), layer=layer
            )
    total += 1
                



 92%|█████████▏| 45966199/49999996 [16:51:08<1:30:04, 746.31it/s] 

In [ ]:
with open('../pickle_files/pretrained_memory_weights2.pickle', 'wb') as f:
    pickle.dump(memory_block, f)

In [199]:
### load pretrained model ###
with open('../pickle_files/pretrained_memory_weights2.pickle', 'rb') as f:
    memory_block = pickle.load(f)

### make prediction block ###
lr_prediction = 1e-3
prediction_block = {}
prediction_criterion = nn.CrossEntropyLoss()

for layer in range(total_layers):
    
    if layer == 0:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, vocab_size, context_size=context_size)
    elif layer == total_layers-1:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, context_size)
    else:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, context_size, context_size=context_size)

prediction_optimizer = torch.optim.Adam(
    itertools.chain(*[prediction_block[layer].parameters() for layer in range(total_layers)]),
    lr=lr_prediction,
    weight_decay=1e-8
)
### handle data ###
total_samples = 100000
n_community = 2
n_members = 3

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)
data_set = Dataset_converter(data, 1, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
context = {}
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    for layer in range(total_layers):
        downsample = short_term_memory**layer

        if layer == 0:
            h[layer] = train_layer(memory_block[layer], memory_optimizers[layer], memory_criteria[layer], X)

        elif total%(downsample)==0:
            input_buffer[layer].append(
                h[layer-1][0][0].clone()
            )

            h[layer] = train_layer(
                memory_block[layer], memory_optimizers[layer], memory_criteria[layer], 
                torch.stack(list(input_buffer[layer])).unsqueeze(0), layer=layer
            )
    
    ### train prediction block ###
    prediction_optimizer.zero_grad()
    for layer in range(total_layers-1,-1,-1):
        if layer == total_layers-1:
            context[layer] = prediction_block[layer](h[layer])
        else:
            context[layer] = prediction_block[layer](h[layer], context[layer+1])

    loss = prediction_criterion(context[0][0][0], y[0][0]) 
    loss.backward()
    prediction_optimizer.step()

    with torch.no_grad():
        total += 1
        if y[0][0] == context[0].argmax():
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0
        
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {np.sum(correct)/total if total<1000 else np.sum(correct)/1000:.4f}')

                


Iter : 1001, loss: 0.6263, accuracy: 0.4420
Iter : 2001, loss: 0.3003, accuracy: 0.6320
Iter : 3001, loss: 0.2313, accuracy: 0.6540
Iter : 4001, loss: 0.2050, accuracy: 0.6720
Iter : 5001, loss: 0.1528, accuracy: 0.6700
Iter : 6001, loss: 0.0756, accuracy: 0.6810
Iter : 7001, loss: 0.0319, accuracy: 0.6640
Iter : 8001, loss: 0.0924, accuracy: 0.6730
Iter : 9001, loss: 0.0150, accuracy: 0.6840
Iter : 10001, loss: 0.0373, accuracy: 0.6730
Iter : 11001, loss: 0.0134, accuracy: 0.6560
Iter : 12001, loss: 0.0237, accuracy: 0.6680
Iter : 13001, loss: 0.0245, accuracy: 0.6820
Iter : 14001, loss: 0.0037, accuracy: 0.6730
Iter : 15001, loss: 0.0033, accuracy: 0.6730
Iter : 16001, loss: 0.0063, accuracy: 0.6610
Iter : 17001, loss: 0.0054, accuracy: 0.6680
Iter : 18001, loss: 0.0123, accuracy: 0.6700
Iter : 19001, loss: 0.0009, accuracy: 0.6700
Iter : 20001, loss: 0.0029, accuracy: 0.6750
Iter : 21001, loss: 0.0011, accuracy: 0.6800
Iter : 22001, loss: 0.0021, accuracy: 0.7070
Iter : 23001, loss: